# Odrive Setup Script

In [ ]:
import odrive
import time

In [ ]:
print("Finding an ODrive...")
odrv0 = odrive.find_any()
print("ODrive found!")

def save_and_reboot(odrv, reconnect=True):
    print("Configuration applied. Saving to ODrive...")
    odrv.save_configuration()
    print("Configuration saved. Rebooting ODrive...")
    odrv.reboot()
    time.sleep(2)  # Wait for ODrive to reboot

    if reconnect:
        print("reconnecting to ODrive...")
        odrv = odrv.find_any()
        print("ODrive reconnected!")
        return odrv
    else:
        print("Not reconnecting to ODrive.")
        return None

### Pre-Configuration

In [ ]:
motor_type = int(input("Enter motor type (6354/5065): "))
max_current = float(input("Enter max current (A): "))
max_supply_current = float(input("Enter max supply current (A): "))

match motor_type:
    case 6354:
        user_config = 'closed_loop_6354.json'
    case 5065:
        user_config = 'closed_loop_5065.json'
    case _:
        print("Invalid motor type. Defaulting to 6354.")
        user_config = 'closed_loop_6354.json'

print(f"Applying {user_config} configuration...")
odrv0.restore_config(user_config)

odrv0 = save_and_reboot(odrv0)

print("Setting current limits...")

odrv0.axis0.motor.config.current_lim = max_current
odrv0.config.dc_max_positive_current = max_supply_current

print("Current limits set. ODrive setup complete!")

odrv0 = save_and_reboot(odrv0)

### Calibration

In [ ]:
odrv0.axis0.requested_state = odrive.enums.AXIS_STATE_FULL_CALIBRATION_SEQUENCE

odrv0.axis0.requested_state = odrive.enums.AXIS_STATE_IDLE

odrv0.axis0.encoder.config.pre_calibrated = True
odrv0.axis0.motor.config.pre_calibrated = True

odrv0 = save_and_reboot(odrv0)

### Test Closed Loop Control

In [ ]:
odrv0.axis0.controller.pos_setpoint = 5.0

### Homing Configuration

In [ ]:
reduction_ratio = float(input('Enter Reduction ratio: '))
endstop_gpio = int(input('Enter endstop gpio: '))
endstop_position = float(input('Enter endstop position: '))